In [7]:
import numpy as np
import pandas as pd

SEED = 42

def load_task(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    feat_cols = [c for c in train.columns if c.startswith("feature_")]
    X = train[feat_cols].to_numpy(dtype=np.float32)
    y = train["label"].to_numpy(dtype=np.int64)
    X_test = test[feat_cols].to_numpy(dtype=np.float32)
    return X, y, X_test, train["id"].to_numpy(), test["id"].to_numpy()

Xi, yi, Xi_test, ids_i, ids_i_test = load_task("Data/image_train.csv", "Data/image_test.csv")
Xs, ys, Xs_test, ids_s, ids_s_test = load_task("Data/sentiment_train.csv", "Data/sentiment_test.csv")

print("image train", Xi.shape, "labels", yi.shape, "test", Xi_test.shape)
print("sentiment train", Xs.shape, "labels", ys.shape, "test", Xs_test.shape)

image train (20000, 500) labels (20000,) test (5000, 500)
sentiment train (12000, 768) labels (12000,) test (5000, 768)


In [12]:
def stratified_split(y, val_frac=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_val = int(round(len(idx) * val_frac))
        val_idx.extend(idx[:n_val])
        train_idx.extend(idx[n_val:])
    return np.sort(train_idx), np.sort(val_idx)

for name, y in [("image", yi), ("sentiment", ys)]:
    tr, va = stratified_split(y)
    pd.DataFrame({
        "index": np.concatenate([tr, va]),
        "split": ["train"] * len(tr) + ["val"] * len(va),
    }).to_csv(f"split_{name}.csv", index=False)
    print(name, len(tr), "train /", len(va), "val")



image 16000 train / 4000 val
sentiment 9600 train / 2400 val
